# Transfer Learning con YOLOv8 para Detección de EPP
Esta libreta ejecuta el pipeline completo de entrenamiento:
1. Descarga del dataset crudo desde Roboflow.
2. Carga del modelo base pre-entrenado (Transfer Learning).
3. Entrenamiento (Fine-tuning) con los datos de seguridad industrial..

---
## 0. Setup

Antes de comenzar debemos poner todas las libretas necesarias para poder trabajar.

In [1]:
import torch
from roboflow import Roboflow
from ultralytics import YOLO
import time
import os
import glob


def get_optimal_device():
    """Detecta el hardware de aceleración disponible según el sistema operativo."""
    if torch.cuda.is_available():
        print("Hardware detectado: CUDA. Utilizando GPU NVIDIA (Windows/Linux).")
        return 'cuda'
    elif torch.backends.mps.is_available():
        print("Hardware detectado: MPS. Utilizando Apple Silicon GPU (Mac).")
        return 'mps'
    else:
        print("Advertencia: No se detectó aceleración por hardware. Utilizando CPU.")
        return 'cpu'

# Asignar el dispositivo óptimo a una variable para usarla en el entrenamiento
device = get_optimal_device()

Hardware detectado: MPS. Utilizando Apple Silicon GPU (Mac).


---
## 1. Descargando los datos

Lo que hacemos dentro de la celda es conectarnos con la API de "roboflow" para poder descargar el conjunto de datos de mi version que puse en roboflow.

In [3]:
# Descarga del dataset desde Roboflow
print("Conectando con Roboflow...")

# Conectarse a Roboflow para descargar los datos
rf = Roboflow(api_key="i51woqHzaLwjGeEuXLFU")

# Hacemos seleccion del espacio de trabajo, este codigo lo genera "roboflow" 
# cuando pides codigo de descarga en una libreta de jupyter
project = rf.workspace("franciscos-workspace-ukujq").project("construction-safety-gsnvb-qzleh")
version = project.version(1)

#Definimos ruta descarga de los datos
dataset_path = "../data/raw/construction_safety"
print("Iniciando descarga del dataset...")

# Descargamos los datos y los guardamos en la ruta que definimos
dataset = version.download("yolov8", location=dataset_path)
print(f"Descarga finalizada. Los datos están listos en: {dataset_path}")

Conectando con Roboflow...
loading Roboflow workspace...
loading Roboflow project...
Iniciando descarga del dataset...
Descarga finalizada. Los datos están listos en: ../data/raw/construction_safety


---
## 2. Cargamos el modelo de YOLO para hacer el transferlerning

**¿Por qué utilizamos `yolov8n.pt`?**

La letra "n" hace referencia a la versión **Nano**. La familia de modelos YOLOv8 viene en distintos tamaños (Nano, Small, Medium, Large, Xtra-large), y hemos seleccionado la versión Nano por las siguientes razones estratégicas:

1. **Optimización para Tiempo Real:** Como el objetivo final del proyecto es realizar inferencias mediante una cámara web en un entorno industrial, necesitamos minimizar la latencia. El modelo Nano es el más rápido de la arquitectura, garantizando una alta tasa de fotogramas por segundo (FPS).
2. **Eficiencia de Recursos:** Al ser el modelo con menos parámetros, requiere menos memoria RAM y poder computacional, lo que lo hace ideal para despliegues ágiles y pruebas iniciales (prototipado rápido).
3. **Transferencia de Conocimiento:** El archivo `.pt` contiene pesos pre-entrenados con el dataset general COCO. Mediante Transfer Learning, aprovechamos la capacidad del modelo para reconocer formas, texturas y bordes básicos, y enfocamos el nuevo entrenamiento (fine-tuning) exclusivamente en que aprenda a diferenciar el Equipo de Protección Personal (EPP).

### ¿Cuál es la función del archivo data.yaml?

Además del cerebro (el modelo pre-entrenado), el algoritmo necesita un mapa. Un archivo **YAML** es simplemente un documento de texto estructurado de forma muy limpia, sin llaves ni código complejo, que funciona como el índice de nuestro dataset. 

Al configurar la variable `yaml_path`, le estamos entregando a YOLO las instrucciones exactas para que pueda arrancar, indicándole dos cosas vitales:
1. **Las rutas:** En qué carpetas exactas de nuestra computadora (`data/raw/`) viven las fotos de entrenamiento y validación.
2. **Las clases:** Cuántos objetos diferentes va a aprender a detectar y cómo se llaman (helmet, vest, person, etc.).

In [5]:

# Cargamos el modelo "yolov8n.pt"
print("Cargando modelo base YOLOv8n para Transfer Learning...")
model = YOLO('yolov8n.pt') 

# Configurando la ruta del archivo yaml_path
yaml_path = f"{dataset_path}/data.yaml"
print(f"Archivo YAML configurado en: {yaml_path}")

Cargando modelo base YOLOv8n para Transfer Learning...
Archivo YAML configurado en: ../data/raw/construction_safety/data.yaml


## 3. Entrenamiento del Modelo (Fine-Tuning)

En esta celda ejecutamos el proceso central de aprendizaje de la red neuronal. Utilizamos la función `train` de Ultralytics, la cual tomará nuestras imágenes etiquetadas de Roboflow y ajustará los pesos del modelo base para que aprenda a detectar cascos, chalecos y personas.

Adicionalmente, se incluye un bloque de medición de tiempo con la librería `time` para hacer un benchmark del rendimiento del hardware durante el procesamiento.

**Explicación de los hiperparámetros:**
* **`data`**: Apunta al archivo `data.yaml`, el cual le indica a YOLO dónde encontrar las imágenes y cuáles son las clases a detectar.
* **`epochs=25`**: Define que el modelo analizará el dataset completo 25 veces. Es una cantidad conservadora y excelente para una primera validación rápida del aprendizaje (sanity check).
* **`imgsz=640`**: Estandariza la resolución de entrada. La arquitectura YOLO está matemáticamente optimizada para procesar matrices cuadradas de 640x640 píxeles.
* **`device`**: Asigna la carga de trabajo al acelerador de hardware detectado previamente (garantizando que se use el GPU y no el procesador central).
* **`project` y `name`**: Controlan la organización de los archivos de salida. Ultralytics creará automáticamente la ruta `../models/yolov8_epp_v1/`, donde guardará las métricas de error, gráficas de precisión y, lo más importante, el archivo `weights/best.pt` con el cerebro final de tu modelo.

In [6]:
print("Iniciando proceso de entrenamiento. Las barras de progreso aparecerán automáticamente...")

# Iniciar el cronómetro
start_time = time.time()

# Esto obliga a YOLO a guardar exactamente en tu carpeta 'models' original
ruta_modelos = os.path.abspath('../models')
print(f"Los resultados se guardarán estrictamente en: {ruta_modelos}")

# Ejecutar el entrenamiento
results = model.train(
    data=yaml_path,
    epochs=25,
    imgsz=640,
    device=device,
    project=ruta_modelos,
    name='yolov8_epp_v1'
)

# Detener el cronómetro
end_time = time.time()

# Calcular el tiempo transcurrido en minutos
elapsed_time_minutes = (end_time - start_time) / 60

print(f"Entrenamiento completado exitosamente.")
print(f"Tiempo total de entrenamiento: {elapsed_time_minutes:.2f} minutos.")

Iniciando proceso de entrenamiento. Las barras de progreso aparecerán automáticamente...
Los resultados se guardarán estrictamente en: /Users/pancakes/Documents/Consultoria/03 Proyectos Tecnicos/epp detection cv/ppe-detection-system/models
New https://pypi.org/project/ultralytics/8.4.18 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.15 🚀 Python-3.11.13 torch-2.10.0 MPS (Apple M4 Pro)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../data/raw/construction_safety/data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=25, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7

In [2]:
print("Buscando el modelo recién entrenado en la carpeta 'models'...")

# Buscamos estrictamente dentro de la carpeta models
archivos_encontrados = glob.glob("../models/**/weights/best.pt", recursive=True)

if archivos_encontrados:
    # Seleccionamos el archivo creado más recientemente
    ruta_modelo_final = max(archivos_encontrados, key=os.path.getctime)
    print(f"¡Modelo encontrado!: {ruta_modelo_final}")
    
    # Cargamos el modelo
    print("Cargando el modelo en memoria...")
    modelo_entrenado = YOLO(ruta_modelo_final)

    # Buscamos la imagen de prueba
    imagenes_prueba = glob.glob("../data/raw/construction_safety/test/images/*.jpg")

    if imagenes_prueba:
        imagen_test = imagenes_prueba[0]
        print(f"Probando modelo con la imagen: {imagen_test}")
        
        # Hacemos la predicción
        resultados = modelo_entrenado.predict(
            source=imagen_test, 
            conf=0.5, 
            save=True, 
            project='../reports', 
            name='prueba_inferencia',
            exist_ok=True # Esto evita que se creen carpetas como prueba_inferencia2, prueba_inferencia3...
        )
        print("¡Inferencia lista! Revisa la carpeta reports/prueba_inferencia/ para ver la foto con las cajas.")
    else:
        print("No se encontraron imágenes de prueba en la ruta indicada.")
else:
    print("Aún no se encuentra el modelo best.pt. Verifica que el entrenamiento haya terminado.")

Buscando el modelo recién entrenado en la carpeta 'models'...
¡Modelo encontrado!: ../models/yolov8_epp_v1/weights/best.pt
Cargando el modelo en memoria...
Probando modelo con la imagen: ../data/raw/construction_safety/test/images/ppe_0120_jpg.rf.12944e3a790011195ce3064b1044940f.jpg

image 1/1 /Users/pancakes/Documents/Consultoria/03 Proyectos Tecnicos/epp detection cv/ppe-detection-system/notebooks/../data/raw/construction_safety/test/images/ppe_0120_jpg.rf.12944e3a790011195ce3064b1044940f.jpg: 640x640 1 helmet, 1 no-vest, 1 person, 33.5ms
Speed: 2.0ms preprocess, 33.5ms inference, 4.7ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /Users/pancakes/Documents/Consultoria/03 Proyectos Tecnicos/epp detection cv/ppe-detection-system/runs/reports/prueba_inferencia
¡Inferencia lista! Revisa la carpeta reports/prueba_inferencia/ para ver la foto con las cajas.


In [5]:
print("Buscando el modelo recién entrenado en la carpeta 'models'...")

# Buscamos estrictamente dentro de la carpeta models
archivos_encontrados = glob.glob("../models/**/weights/best.pt", recursive=True)

if archivos_encontrados:
    # Seleccionamos el archivo creado más recientemente
    ruta_modelo_final = max(archivos_encontrados, key=os.path.getctime)
    print(f"¡Modelo encontrado!: {ruta_modelo_final}")
    
    # Cargamos el modelo
    print("Cargando el modelo en memoria...")
    modelo_entrenado = YOLO(ruta_modelo_final)

    # Buscamos la imagen de prueba
    imagenes_prueba = glob.glob("../data/raw/construction_safety/test/images/*.jpg")

    if imagenes_prueba:
        imagen_test = imagenes_prueba[0:10]
        print(f"Probando modelo con la imagen: {imagen_test}")
        
        #Le damos la ruta de guardado
        ruta_reportes = os.path.abspath('../reports')
        
        # Hacemos la predicción
        resultados = modelo_entrenado.predict(
            source=imagen_test, 
            conf=0.5, 
            save=True, 
            project=ruta_reportes, # <--- Usamos la variable con la ruta absoluta
            name='prueba_inferencia',
            exist_ok=True # Esto evita que se creen carpetas como prueba_inferencia2, prueba_inferencia3...
        )
        print("¡Inferencia lista! Revisa tu carpeta original reports/prueba_inferencia/ para ver la foto con las cajas.")
    else:
        print("No se encontraron imágenes de prueba en la ruta indicada.")
else:
    print("Aún no se encuentra el modelo best.pt. Verifica que el entrenamiento haya terminado.")

Buscando el modelo recién entrenado en la carpeta 'models'...
¡Modelo encontrado!: ../models/yolov8_epp_v1/weights/best.pt
Cargando el modelo en memoria...
Probando modelo con la imagen: ['../data/raw/construction_safety/test/images/ppe_0120_jpg.rf.12944e3a790011195ce3064b1044940f.jpg', '../data/raw/construction_safety/test/images/ppe_0857_jpg.rf.3ab5492afda06d865cc5c2caa6694111.jpg', '../data/raw/construction_safety/test/images/ppe_0342_jpg.rf.57cb3b3a64b1523219e72202b1c0a79e.jpg', '../data/raw/construction_safety/test/images/ppe_0388_jpg.rf.cd174e060d1d4e8f1eadeaf672224758.jpg', '../data/raw/construction_safety/test/images/ppe_0261_jpg.rf.e768ec02a950f4e40cbba4366cd1669b.jpg', '../data/raw/construction_safety/test/images/ppe_0643_jpg.rf.da407c3d00f6d566d744edff412369ff.jpg', '../data/raw/construction_safety/test/images/ppe_0745_jpg.rf.b1b819b673c74b85cbe1e77447f935ed.jpg', '../data/raw/construction_safety/test/images/ppe_0779_jpg.rf.06be31f46467537eeeb6f13df4b7c625.jpg', '../data/raw